# 04g — Siamese V3 con Batch-Hard Triplet Mining

## Obiettivo

Migliorare la separazione multiclass dello spazio embedding appreso dalla
Siamese Network.

La Euclidean Contrastive Loss ha prodotto una buona capacità pairwise:

- validation ROC-AUC ≈ 0.68

ma la classificazione per similarità sulle 18 classi raggiunge soltanto
circa il 24–26% di accuracy.

Questo indica che lo spazio metrico appreso distingue abbastanza bene
coppie same/different, ma non separa ancora sufficientemente le singole
classi di malattia.

In questo esperimento viene effettuato un fine-tuning dell'encoder
Euclidean V3 utilizzando Batch-Hard Triplet Loss.

Per ogni anchor vengono selezionati all'interno del batch:

- hardest positive: sample della stessa classe più distante;
- hardest negative: sample di classe diversa più vicino.

L'obiettivo è ottenere cluster intra-classe più compatti e aumentare la
separazione inter-classe.

## Configurazione iniziale

- FCGR k=6
- CNN V3 bias-free
- embedding 128D L2-normalizzato
- inizializzazione dal best checkpoint Euclidean V3
- batch class-balanced
- 18 classi
- selezione checkpoint tramite validation Macro-F1 multiclass
- test set non utilizzato durante il tuning

In [1]:
# ============================================================
# CELL 2 — IMPORT
# ============================================================

from pathlib import Path

import json
import random
import time
import copy
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    Sampler
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)


print(
    "PyTorch:",
    torch.__version__
)

print(
    "CUDA disponibile:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

PyTorch: 2.12.0+cu126
CUDA disponibile: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# ============================================================
# CELL 3 — PATH E CONFIGURAZIONE
# ============================================================

CURRENT_DIR = (
    Path.cwd()
    .resolve()
)


if CURRENT_DIR.name == "notebooks":

    PROJECT_ROOT = (
        CURRENT_DIR.parent
    )

else:

    PROJECT_ROOT = (
        CURRENT_DIR
    )


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
)


TRIPLET_ARTIFACTS_DIR = (
    ARTIFACTS_DIR
    / "siamese_triplet_hard"
)


TRIPLET_ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# DATA
# ============================================================

PAIR_CONFIG_PATH = (
    PROCESSED_DIR
    / "siamese_pair_config.json"
)


MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_primary_manifest.tsv"
)


VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)


TEST_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_test_pair_pool.tsv"
)


with open(
    PAIR_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    PAIR_CONFIG = json.load(f)


K = int(
    PAIR_CONFIG["k"]
)


RANDOM_STATE = int(
    PAIR_CONFIG["random_state"]
)


EMBEDDING_DIM = 128


FCGR_MEMMAP_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}.npy"
)


FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}_index.tsv"
)


# ============================================================
# PRETRAINED EUCLIDEAN CHECKPOINT
# ============================================================

PRETRAINED_CHECKPOINT_PATH = (
    ARTIFACTS_DIR
    / "siamese_euclidean_v3"
    / "euclidean_v3_margin_1p25_best.pt"
)


assert PRETRAINED_CHECKPOINT_PATH.exists()


# ============================================================
# TRIPLET CONFIG
# ============================================================

N_CLASSES_PER_BATCH = 18

N_SAMPLES_PER_CLASS = 7

TRIPLET_BATCH_SIZE = (
    N_CLASSES_PER_BATCH
    *
    N_SAMPLES_PER_CLASS
)


TRIPLET_MARGIN = 0.20

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 40

EARLY_STOPPING_PATIENCE = 8

MIN_DELTA = 1e-4


print(
    "Triplet batch size:",
    TRIPLET_BATCH_SIZE
)

print(
    "Margin:",
    TRIPLET_MARGIN
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Checkpoint iniziale:",
    PRETRAINED_CHECKPOINT_PATH
)

Triplet batch size: 126
Margin: 0.2
Learning rate: 0.0001
Checkpoint iniziale: D:\Daria\Desktop\eccdna_fcgr_siamese\artifacts\siamese_euclidean_v3\euclidean_v3_margin_1p25_best.pt


In [3]:
# ============================================================
# CELL 4 — DEVICE E DATA
# ============================================================

def set_seed(
    seed
):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(
    RANDOM_STATE
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )


# ============================================================
# METADATA
# ============================================================

metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


val_metadata = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


train_metadata = (
    metadata[
        metadata[
            "split_cluster"
        ] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)


train_metadata[
    "class_id"
] = (
    train_metadata[
        "class_id"
    ]
    .astype(int)
)


val_metadata[
    "class_id"
] = (
    val_metadata[
        "class_id"
    ]
    .astype(int)
)


# ============================================================
# FCGR
# ============================================================

fcgr_memmap = np.load(
    FCGR_MEMMAP_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={
        "id": str
    }
)


id_to_fcgr_row = dict(
    zip(
        fcgr_index["id"],
        fcgr_index["fcgr_row"]
    )
)


print(
    "Device:",
    DEVICE
)

print(
    "Train samples:",
    len(train_metadata)
)

print(
    "Val samples:",
    len(val_metadata)
)

print(
    "Numero classi train:",
    train_metadata[
        "class_id"
    ].nunique()
)

print(
    "FCGR:",
    fcgr_memmap.shape
)

Device: cuda
Train samples: 126265
Val samples: 12937
Numero classi train: 18
FCGR: (150272, 64, 64)


In [4]:
# ============================================================
# CELL 5 — ENCODER V3
# ============================================================

class FCGRCNNEncoderV3(
    nn.Module
):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()


        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                32
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv2d(
                32,
                32,
                3,
                padding=1,
                bias=False
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                32,
                64,
                3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                64,
                128,
                3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                128,
                128,
                3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.AdaptiveAvgPool2d(
                (4, 4)
            )
        )


        self.embedding_head = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(
                256,
                embedding_dim
            )
        )


    def forward(
        self,
        x
    ):

        x = self.features(
            x
        )

        z = self.embedding_head(
            x
        )

        return F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8
        )


class SiameseNetworkV3(
    nn.Module
):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()

        self.encoder = (
            FCGRCNNEncoderV3(
                embedding_dim
            )
        )


    def forward(
        self,
        x
    ):

        return self.encoder(
            x
        )

In [5]:
# ============================================================
# CELL 6 — LOAD PRETRAINED EUCLIDEAN V3
# ============================================================

pretrained_checkpoint = torch.load(
    PRETRAINED_CHECKPOINT_PATH,
    map_location=DEVICE
)


triplet_model = (
    SiameseNetworkV3(
        embedding_dim=
            EMBEDDING_DIM
    )
    .to(
        DEVICE
    )
)


triplet_model.load_state_dict(
    pretrained_checkpoint[
        "model_state_dict"
    ],
    strict=True
)


print("=" * 70)
print("PRETRAINED MODEL LOADED")
print("=" * 70)

print(
    "Source best epoch:",
    pretrained_checkpoint[
        "best_epoch"
    ]
)

print(
    "Source val AUC:",
    pretrained_checkpoint[
        "best_val_auc"
    ]
)


# ============================================================
# SANITY CHECK
# ============================================================

batch_rows = (
    train_metadata
    .head(8)
)


x_check = []


for sample_id in batch_rows["id"]:

    row = id_to_fcgr_row[
        str(sample_id)
    ]

    fcgr = np.array(
        fcgr_memmap[row],
        dtype=np.float32,
        copy=True
    )

    x_check.append(
        fcgr
    )


x_check = torch.from_numpy(
    np.stack(
        x_check
    )
).unsqueeze(1).to(
    DEVICE
)


triplet_model.eval()


with torch.no_grad():

    z_check = triplet_model(
        x_check
    )


print(
    "Embedding shape:",
    z_check.shape
)

print(
    "Norma media:",
    z_check.norm(
        dim=1
    ).mean().item()
)


assert (
    z_check.shape
    ==
    (
        8,
        EMBEDDING_DIM
    )
)


assert torch.allclose(

    z_check.norm(
        dim=1
    ),

    torch.ones(
        8,
        device=DEVICE
    ),

    atol=1e-4
)


print(
    "Checkpoint compatibile: OK"
)


del x_check
del z_check

gc.collect()

torch.cuda.empty_cache()

PRETRAINED MODEL LOADED
Source best epoch: 38
Source val AUC: 0.67727181092637
Embedding shape: torch.Size([8, 128])
Norma media: 1.0
Checkpoint compatibile: OK


In [6]:
# ============================================================
# CELL 7 — SINGLE FCGR DATASET
# ============================================================

class SingleFCGRDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row
    ):

        self.metadata = (
            metadata[
                [
                    "id",
                    "class_id"
                ]
            ]
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = (
            fcgr_memmap
        )


        self.rows = (
            self.metadata[
                "id"
            ]
            .map(
                id_to_row
            )
            .to_numpy(
                dtype=np.int64
            )
        )


        self.labels = (
            self.metadata[
                "class_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )


    def __len__(
        self
    ):

        return len(
            self.metadata
        )


    def __getitem__(
        self,
        index
    ):

        row = int(
            self.rows[index]
        )


        fcgr = np.array(
            self.fcgr_memmap[row],
            dtype=np.float32,
            copy=True
        )


        return {

            "x":
                torch.from_numpy(
                    fcgr
                )
                .unsqueeze(0),

            "class_id":
                torch.tensor(
                    self.labels[index],
                    dtype=torch.long
                )
        }


train_dataset = SingleFCGRDataset(

    metadata=
        train_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


print(
    "Train dataset:",
    len(train_dataset)
)

print(
    "Numero classi:",
    len(
        np.unique(
            train_dataset.labels
        )
    )
)

Train dataset: 126265
Numero classi: 18


In [7]:
# ============================================================
# CELL 8 — PK BATCH SAMPLER
# ============================================================

class PKBatchSampler(Sampler):

    def __init__(
        self,
        labels,
        n_classes_per_batch,
        n_samples_per_class,
        seed=42
    ):

        self.labels = np.asarray(
            labels,
            dtype=np.int64
        )


        self.classes = np.array(
            sorted(
                np.unique(
                    self.labels
                )
            ),
            dtype=np.int64
        )


        self.P = int(
            n_classes_per_batch
        )

        self.K = int(
            n_samples_per_class
        )

        self.seed = int(
            seed
        )


        if self.P > len(self.classes):

            raise ValueError(
                "P maggiore del numero "
                "di classi disponibili."
            )


        # ====================================================
        # CLASS -> DATASET INDICES
        # ====================================================

        self.class_to_indices = {

            int(class_id):
                np.where(
                    self.labels
                    ==
                    class_id
                )[0]

            for class_id
            in self.classes
        }


        # ====================================================
        # DOWN-SAMPLING PER EPOCH
        #
        # Usiamo come riferimento la classe meno numerosa.
        # Nessuna classe viene quindi oversamplata
        # all'interno della stessa epoca.
        # ====================================================

        self.min_class_size = min(

            len(indices)

            for indices
            in self.class_to_indices.values()
        )


        self.batches_per_epoch = (
            self.min_class_size
            //
            self.K
        )


        self.epoch = 0


    def __len__(
        self
    ):

        return (
            self.batches_per_epoch
        )


    def __iter__(
        self
    ):

        rng = np.random.default_rng(
            self.seed
            +
            self.epoch
        )


        # Shuffle indipendente dei sample
        # di ciascuna classe.
        shuffled_indices = {

            int(class_id):
                rng.permutation(
                    self.class_to_indices[
                        int(class_id)
                    ]
                )

            for class_id
            in self.classes
        }


        pointers = {

            int(class_id): 0

            for class_id
            in self.classes
        }


        for _ in range(
            self.batches_per_epoch
        ):

            # ------------------------------------------------
            # Nel nostro caso P = 18,
            # quindi tutte le classi sono presenti.
            # ------------------------------------------------

            if (
                self.P
                ==
                len(self.classes)
            ):

                batch_classes = (
                    self.classes
                )

            else:

                batch_classes = rng.choice(
                    self.classes,
                    size=self.P,
                    replace=False
                )


            batch_indices = []


            for class_id in batch_classes:

                class_id = int(
                    class_id
                )


                start = (
                    pointers[
                        class_id
                    ]
                )

                end = (
                    start
                    +
                    self.K
                )


                selected = (
                    shuffled_indices[
                        class_id
                    ][
                        start:end
                    ]
                )


                # Questa condizione non dovrebbe verificarsi
                # con P=18 e batches_per_epoch basato
                # sulla classe più piccola.
                if (
                    len(selected)
                    <
                    self.K
                ):

                    selected = rng.choice(
                        self.class_to_indices[
                            class_id
                        ],
                        size=self.K,
                        replace=False
                    )


                pointers[
                    class_id
                ] = end


                batch_indices.extend(
                    selected.tolist()
                )


            # Mischiamo la posizione delle classi
            # all'interno del batch.
            rng.shuffle(
                batch_indices
            )


            yield batch_indices


        # Epoca successiva:
        # shuffle differente ma riproducibile.
        self.epoch += 1

In [8]:
# ============================================================
# CELL 9 — PK DATALOADER SANITY CHECK
# ============================================================

train_pk_sampler = PKBatchSampler(

    labels=
        train_dataset.labels,

    n_classes_per_batch=
        N_CLASSES_PER_BATCH,

    n_samples_per_class=
        N_SAMPLES_PER_CLASS,

    seed=
        RANDOM_STATE
)


train_loader = DataLoader(

    train_dataset,

    batch_sampler=
        train_pk_sampler,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


print("=" * 70)
print("PK SAMPLER")
print("=" * 70)

print(
    "Classi per batch:",
    N_CLASSES_PER_BATCH
)

print(
    "Sample per classe:",
    N_SAMPLES_PER_CLASS
)

print(
    "Batch size:",
    TRIPLET_BATCH_SIZE
)

print(
    "Classe più piccola:",
    train_pk_sampler.min_class_size
)

print(
    "Batch per epoca:",
    len(train_pk_sampler)
)


# ============================================================
# CHECK PRIMO BATCH
# ============================================================

batch_check = next(
    iter(train_loader)
)


x_check = (
    batch_check[
        "x"
    ]
)


y_check = (
    batch_check[
        "class_id"
    ]
)


unique_classes, counts = torch.unique(
    y_check,
    return_counts=True
)


print()
print(
    "x shape:",
    x_check.shape
)

print(
    "y shape:",
    y_check.shape
)

print(
    "Numero classi nel batch:",
    len(unique_classes)
)

print(
    "Conteggi per classe:",
    counts.tolist()
)


assert (
    x_check.shape[0]
    ==
    TRIPLET_BATCH_SIZE
)


assert (
    len(unique_classes)
    ==
    N_CLASSES_PER_BATCH
)


assert torch.all(
    counts
    ==
    N_SAMPLES_PER_CLASS
)


print()
print(
    "PK batch corretto: OK"
)

PK SAMPLER
Classi per batch: 18
Sample per classe: 7
Batch size: 126
Classe più piccola: 1755
Batch per epoca: 250

x shape: torch.Size([126, 1, 64, 64])
y shape: torch.Size([126])
Numero classi nel batch: 18
Conteggi per classe: [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7]

PK batch corretto: OK


In [9]:
# ============================================================
# CELL 10 — BATCH-HARD TRIPLET LOSS
# ============================================================

class BatchHardTripletLoss(nn.Module):

    def __init__(
        self,
        margin=0.20
    ):

        super().__init__()

        self.margin = float(
            margin
        )


    def forward(
        self,
        embeddings,
        labels
    ):

        # ====================================================
        # FP32 per stabilità numerica della matrice distanze
        # ====================================================

        embeddings = (
            embeddings.float()
        )


        labels = (
            labels.long()
        )


        # ====================================================
        # PAIRWISE DISTANCE MATRIX
        #
        # shape:
        # [batch, batch]
        # ====================================================

        distance_matrix = torch.cdist(
            embeddings,
            embeddings,
            p=2
        )


        batch_size = (
            labels.shape[0]
        )


        # ====================================================
        # MASK SAME CLASS / DIFFERENT CLASS
        # ====================================================

        same_class = (
            labels.unsqueeze(0)
            ==
            labels.unsqueeze(1)
        )


        identity = torch.eye(
            batch_size,
            dtype=torch.bool,
            device=labels.device
        )


        # Positivi:
        # stessa classe ma NON se stesso.
        positive_mask = (
            same_class
            &
            ~identity
        )


        # Negativi:
        # classe diversa.
        negative_mask = (
            ~same_class
        )


        # ====================================================
        # HARDEST POSITIVE
        #
        # distanza MASSIMA fra sample stessa classe
        # ====================================================

        positive_distances = (
            distance_matrix
            .masked_fill(
                ~positive_mask,
                float("-inf")
            )
        )


        hardest_positive = (
            positive_distances
            .max(
                dim=1
            )
            .values
        )


        # ====================================================
        # HARDEST NEGATIVE
        #
        # distanza MINIMA fra sample classe diversa
        # ====================================================

        negative_distances = (
            distance_matrix
            .masked_fill(
                ~negative_mask,
                float("inf")
            )
        )


        hardest_negative = (
            negative_distances
            .min(
                dim=1
            )
            .values
        )


        # ====================================================
        # TRIPLET LOSS
        # ====================================================

        losses = F.relu(

            hardest_positive
            -
            hardest_negative
            +
            self.margin
        )


        loss = (
            losses.mean()
        )


        # ====================================================
        # DIAGNOSTICHE
        # ====================================================

        active_fraction = (
            (
                losses > 0
            )
            .float()
            .mean()
        )


        hard_positive_mean = (
            hardest_positive
            .mean()
        )


        hard_negative_mean = (
            hardest_negative
            .mean()
        )


        separation = (
            hard_negative_mean
            -
            hard_positive_mean
        )


        stats = {

            "hard_positive":
                hard_positive_mean.detach(),

            "hard_negative":
                hard_negative_mean.detach(),

            "separation":
                separation.detach(),

            "active_fraction":
                active_fraction.detach()
        }


        return (
            loss,
            stats
        )


triplet_criterion = (
    BatchHardTripletLoss(
        margin=
            TRIPLET_MARGIN
    )
)


print(
    "Triplet margin:",
    triplet_criterion.margin
)

Triplet margin: 0.2


In [10]:
# ============================================================
# CELL 11 — TRIPLET LOSS SANITY CHECK
# ============================================================

triplet_model.eval()


batch_check = next(
    iter(train_loader)
)


x = (
    batch_check[
        "x"
    ]
    .to(
        DEVICE,
        non_blocking=True
    )
)


labels = (
    batch_check[
        "class_id"
    ]
    .to(
        DEVICE,
        non_blocking=True
    )
)


with torch.no_grad():

    with torch.autocast(

        device_type=
            DEVICE.type,

        dtype=
            torch.float16
            if DEVICE.type == "cuda"
            else torch.bfloat16,

        enabled=
            DEVICE.type == "cuda"

    ):

        embeddings = (
            triplet_model(
                x
            )
        )


    (
        triplet_loss_check,
        triplet_stats_check
    ) = triplet_criterion(
        embeddings,
        labels
    )


print("=" * 70)
print("BATCH-HARD TRIPLET — PRETRAINED SPACE")
print("=" * 70)


print(
    "Loss:",
    f"{triplet_loss_check.item():.6f}"
)

print(
    "Hard positive:",
    f"{triplet_stats_check['hard_positive'].item():.4f}"
)

print(
    "Hard negative:",
    f"{triplet_stats_check['hard_negative'].item():.4f}"
)

print(
    "Separation:",
    f"{triplet_stats_check['separation'].item():.4f}"
)

print(
    "Active triplets:",
    f"{triplet_stats_check['active_fraction'].item() * 100:.2f}%"
)


print()
print(
    "Embedding norm mean:",
    embeddings
    .norm(dim=1)
    .mean()
    .item()
)


assert torch.isfinite(
    triplet_loss_check
)


assert (
    0.0
    <=
    triplet_stats_check[
        "active_fraction"
    ].item()
    <=
    1.0
)


del x
del labels
del embeddings

gc.collect()

torch.cuda.empty_cache()

BATCH-HARD TRIPLET — PRETRAINED SPACE
Loss: 0.695971
Hard positive: 0.7154
Hard negative: 0.2195
Separation: -0.4960
Active triplets: 100.00%

Embedding norm mean: 1.0


In [11]:
# ============================================================
# CELL 12 — FIXED REFERENCE SET + VALIDATION DATASET
# ============================================================

N_REFERENCE_PER_CLASS = 500

VALIDATION_REFERENCE_SEED = (
    RANDOM_STATE
    +
    80_000
)


rng_reference = np.random.default_rng(
    VALIDATION_REFERENCE_SEED
)


reference_parts = []


for class_id, group in train_metadata.groupby(
    "class_id"
):

    if len(group) < N_REFERENCE_PER_CLASS:

        raise ValueError(
            f"Classe {class_id}: "
            f"solo {len(group)} sample."
        )


    selected_indices = rng_reference.choice(
        len(group),
        size=N_REFERENCE_PER_CLASS,
        replace=False
    )


    reference_parts.append(
        group.iloc[
            selected_indices
        ]
    )


reference_metadata = (
    pd.concat(
        reference_parts,
        ignore_index=True
    )
)


# ============================================================
# DATASETS
# ============================================================

reference_dataset = SingleFCGRDataset(

    metadata=
        reference_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


val_dataset = SingleFCGRDataset(

    metadata=
        val_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


# ============================================================
# LOADERS
# ============================================================

EVAL_BATCH_SIZE = 256


reference_loader = DataLoader(

    reference_dataset,

    batch_size=
        EVAL_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


val_loader = DataLoader(

    val_dataset,

    batch_size=
        EVAL_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


print(
    "Reference samples:",
    len(reference_dataset)
)

print(
    "Reference per classe:",
    N_REFERENCE_PER_CLASS
)

print(
    "Validation samples:",
    len(val_dataset)
)

print(
    "Reference batches:",
    len(reference_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

Reference samples: 9000
Reference per classe: 500
Validation samples: 12937
Reference batches: 36
Validation batches: 51


In [12]:
# ============================================================
# CELL 13 — EXTRACT EMBEDDINGS
# ============================================================

def extract_embeddings(
    model,
    loader
):

    model.eval()


    embeddings_list = []
    labels_list = []


    with torch.no_grad():

        for batch in loader:

            x = (
                batch["x"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            labels = (
                batch["class_id"]
                .cpu()
                .numpy()
            )


            with torch.autocast(

                device_type=
                    DEVICE.type,

                dtype=
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16,

                enabled=
                    DEVICE.type == "cuda"

            ):

                embeddings = model(
                    x
                )


            embeddings_list.append(
                embeddings
                .float()
                .cpu()
                .numpy()
            )


            labels_list.append(
                labels
            )


    return (
        np.concatenate(
            embeddings_list,
            axis=0
        ),
        np.concatenate(
            labels_list,
            axis=0
        ).astype(
            np.int64
        )
    )

In [13]:
# ============================================================
# CELL 14 — METRIC VALIDATION
# ============================================================

def evaluate_multiclass_metric_space(
    model,
    reference_loader,
    val_loader
):

    # ========================================================
    # REFERENCE EMBEDDINGS
    # ========================================================

    (
        reference_embeddings,
        reference_labels
    ) = extract_embeddings(
        model,
        reference_loader
    )


    (
        val_embeddings,
        val_labels
    ) = extract_embeddings(
        model,
        val_loader
    )


    classes = np.array(
        sorted(
            np.unique(
                reference_labels
            )
        ),
        dtype=np.int64
    )


    # ========================================================
    # CLASS PROTOTYPES
    # ========================================================

    prototypes = []


    for class_id in classes:

        class_embeddings = (
            reference_embeddings[
                reference_labels
                ==
                class_id
            ]
        )


        prototype = (
            class_embeddings
            .mean(
                axis=0
            )
        )


        prototype = (
            prototype
            /
            (
                np.linalg.norm(
                    prototype
                )
                +
                1e-12
            )
        )


        prototypes.append(
            prototype
        )


    prototypes = np.stack(
        prototypes,
        axis=0
    ).astype(
        np.float32
    )


    # ========================================================
    # EUCLIDEAN DISTANCES
    # ========================================================

    distances = np.sqrt(
        (
            (
                val_embeddings[
                    :,
                    None,
                    :
                ]
                -
                prototypes[
                    None,
                    :,
                    :
                ]
            )
            ** 2
        )
        .sum(
            axis=2
        )
    )


    prediction_index = (
        distances.argmin(
            axis=1
        )
    )


    y_pred = (
        classes[
            prediction_index
        ]
    )


    # ========================================================
    # METRICS
    # ========================================================

    metrics = {

        "accuracy":
            float(
                accuracy_score(
                    val_labels,
                    y_pred
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    val_labels,
                    y_pred,
                    average="macro",
                    zero_division=0
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    val_labels,
                    y_pred
                )
            )
    }


    return (
        metrics,
        val_labels,
        y_pred
    )

In [14]:
# ============================================================
# CELL 15 — PRE-TRIPLET MULTICLASS BASELINE
# ============================================================

print(
    "Calcolo baseline multiclass..."
)


(
    baseline_multiclass_metrics,
    baseline_y_true,
    baseline_y_pred
) = evaluate_multiclass_metric_space(

    model=
        triplet_model,

    reference_loader=
        reference_loader,

    val_loader=
        val_loader
)


print()
print("=" * 70)
print("PRE-TRIPLET BASELINE")
print("=" * 70)

print(
    "Accuracy:",
    f"{baseline_multiclass_metrics['accuracy']:.4f}"
)

print(
    "Macro-F1:",
    f"{baseline_multiclass_metrics['macro_f1']:.4f}"
)

print(
    "Balanced Accuracy:",
    f"{baseline_multiclass_metrics['balanced_accuracy']:.4f}"
)

Calcolo baseline multiclass...

PRE-TRIPLET BASELINE
Accuracy: 0.2353
Macro-F1: 0.1857
Balanced Accuracy: 0.2402


In [15]:
# ============================================================
# CELL 16 — OPTIMIZER + AMP
# ============================================================

AMP_ENABLED = (
    DEVICE.type == "cuda"
)


try:

    optimizer = torch.optim.AdamW(

        triplet_model.parameters(),

        lr=
            LEARNING_RATE,

        weight_decay=
            WEIGHT_DECAY,

        fused=
            DEVICE.type == "cuda"
    )

    fused_adamw = (
        DEVICE.type == "cuda"
    )


except (
    TypeError,
    RuntimeError
):

    optimizer = torch.optim.AdamW(

        triplet_model.parameters(),

        lr=
            LEARNING_RATE,

        weight_decay=
            WEIGHT_DECAY
    )

    fused_adamw = False


# API moderna, evita il FutureWarning
scaler = torch.amp.GradScaler(

    "cuda",

    enabled=
        AMP_ENABLED
)


print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "AMP:",
    AMP_ENABLED
)

print(
    "Fused AdamW:",
    fused_adamw
)

Learning rate: 0.0001
Weight decay: 0.0001
AMP: True
Fused AdamW: True


In [16]:
# ============================================================
# CELL 17 — TRAIN ONE TRIPLET EPOCH
# ============================================================

def train_triplet_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler
):

    model.train()


    total_loss = 0.0

    total_hard_positive = 0.0

    total_hard_negative = 0.0

    total_separation = 0.0

    total_active_fraction = 0.0

    total_batches = 0


    start_time = time.perf_counter()


    for batch in loader:

        x = (
            batch["x"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        labels = (
            batch["class_id"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(

            device_type=
                DEVICE.type,

            dtype=
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16,

            enabled=
                AMP_ENABLED

        ):

            embeddings = model(
                x
            )


        # La loss internamente forza FP32
        (
            loss,
            stats
        ) = criterion(
            embeddings,
            labels
        )


        if AMP_ENABLED:

            scaler.scale(
                loss
            ).backward()


            scaler.step(
                optimizer
            )


            scaler.update()


        else:

            loss.backward()

            optimizer.step()


        total_loss += (
            loss.detach().item()
        )


        total_hard_positive += (
            stats[
                "hard_positive"
            ].item()
        )


        total_hard_negative += (
            stats[
                "hard_negative"
            ].item()
        )


        total_separation += (
            stats[
                "separation"
            ].item()
        )


        total_active_fraction += (
            stats[
                "active_fraction"
            ].item()
        )


        total_batches += 1


    elapsed = (
        time.perf_counter()
        -
        start_time
    )


    return {

        "loss":
            total_loss
            /
            total_batches,

        "hard_positive":
            total_hard_positive
            /
            total_batches,

        "hard_negative":
            total_hard_negative
            /
            total_batches,

        "separation":
            total_separation
            /
            total_batches,

        "active_fraction":
            total_active_fraction
            /
            total_batches,

        "seconds":
            elapsed
    }

In [17]:
# ============================================================
# CELL 18 — 3-EPOCH SMOKE TEST
# ============================================================

SMOKE_EPOCHS = 3


# Backup stato iniziale
initial_model_state = copy.deepcopy(
    triplet_model.state_dict()
)


initial_optimizer_state = copy.deepcopy(
    optimizer.state_dict()
)


print("=" * 76)
print("BATCH-HARD TRIPLET — SMOKE TEST")
print("=" * 76)


for epoch in range(
    1,
    SMOKE_EPOCHS + 1
):

    train_metrics = train_triplet_epoch(

        model=
            triplet_model,

        loader=
            train_loader,

        criterion=
            triplet_criterion,

        optimizer=
            optimizer,

        scaler=
            scaler
    )


    (
        val_metrics,
        _,
        _
    ) = evaluate_multiclass_metric_space(

        model=
            triplet_model,

        reference_loader=
            reference_loader,

        val_loader=
            val_loader
    )


    print(

        f"Epoch {epoch:02d}/{SMOKE_EPOCHS}"

        f" | loss "
        f"{train_metrics['loss']:.4f}"

        f" | hard+ "
        f"{train_metrics['hard_positive']:.4f}"

        f" | hard- "
        f"{train_metrics['hard_negative']:.4f}"

        f" | sep "
        f"{train_metrics['separation']:.4f}"

        f" | active "
        f"{train_metrics['active_fraction'] * 100:.1f}%"

        f" | val Acc "
        f"{val_metrics['accuracy']:.4f}"

        f" | val F1 "
        f"{val_metrics['macro_f1']:.4f}"

        f" | bal Acc "
        f"{val_metrics['balanced_accuracy']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

BATCH-HARD TRIPLET — SMOKE TEST
Epoch 01/3 | loss 0.3443 | hard+ 0.2276 | hard- 0.0833 | sep -0.1443 | active 100.0% | val Acc 0.1914 | val F1 0.1616 | bal Acc 0.1998 | 15.3s
Epoch 02/3 | loss 0.2088 | hard+ 0.0142 | hard- 0.0054 | sep -0.0088 | active 100.0% | val Acc 0.1953 | val F1 0.1614 | bal Acc 0.2006 | 6.1s
Epoch 03/3 | loss 0.2031 | hard+ 0.0052 | hard- 0.0021 | sep -0.0031 | active 100.0% | val Acc 0.1864 | val F1 0.1572 | bal Acc 0.1931 | 6.1s


In [18]:
# ============================================================
# CELL 19 — RESTORE PRE-TRIPLET MODEL
# ============================================================

triplet_model.load_state_dict(
    initial_model_state
)


optimizer.load_state_dict(
    initial_optimizer_state
)


# Reset GradScaler
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


triplet_model.eval()


print(
    "Modello ripristinato al checkpoint Euclidean V3."
)


# ============================================================
# CHECK GEOMETRY
# ============================================================

batch_restore = next(
    iter(train_loader)
)


x_restore = (
    batch_restore["x"]
    .to(
        DEVICE,
        non_blocking=True
    )
)


y_restore = (
    batch_restore["class_id"]
    .to(
        DEVICE,
        non_blocking=True
    )
)


with torch.no_grad():

    z_restore = triplet_model(
        x_restore
    )


    (
        restore_loss,
        restore_stats
    ) = triplet_criterion(
        z_restore,
        y_restore
    )


print(
    "Hard positive:",
    f"{restore_stats['hard_positive'].item():.4f}"
)

print(
    "Hard negative:",
    f"{restore_stats['hard_negative'].item():.4f}"
)

print(
    "Separation:",
    f"{restore_stats['separation'].item():.4f}"
)

print(
    "Norm embedding:",
    f"{z_restore.norm(dim=1).mean().item():.4f}"
)


del x_restore
del y_restore
del z_restore

gc.collect()
torch.cuda.empty_cache()

Modello ripristinato al checkpoint Euclidean V3.
Hard positive: 0.7159
Hard negative: 0.2292
Separation: -0.4867
Norm embedding: 1.0000


In [19]:
# ============================================================
# CELL 20 — BALANCED IN-BATCH CONTRASTIVE LOSS
# ============================================================

class BalancedInBatchContrastiveLoss(nn.Module):

    def __init__(
        self,
        margin=1.25
    ):

        super().__init__()

        self.margin = float(
            margin
        )


    def forward(
        self,
        embeddings,
        labels
    ):

        embeddings = embeddings.float()

        labels = labels.long()


        distances = torch.cdist(
            embeddings,
            embeddings,
            p=2
        )


        n = labels.shape[0]


        same_class = (
            labels.unsqueeze(0)
            ==
            labels.unsqueeze(1)
        )


        # Usiamo soltanto triangolo superiore:
        # ogni coppia viene considerata una volta.
        upper_triangle = torch.triu(
            torch.ones(
                n,
                n,
                dtype=torch.bool,
                device=labels.device
            ),
            diagonal=1
        )


        positive_mask = (
            same_class
            &
            upper_triangle
        )


        negative_mask = (
            (~same_class)
            &
            upper_triangle
        )


        positive_distances = (
            distances[
                positive_mask
            ]
        )


        negative_distances = (
            distances[
                negative_mask
            ]
        )


        # Stessa logica della nostra
        # Euclidean Contrastive Loss.
        positive_loss = (
            positive_distances
            .pow(2)
            .mean()
        )


        negative_loss = (
            F.relu(
                self.margin
                -
                negative_distances
            )
            .pow(2)
            .mean()
        )


        # Separiamo le medie per non far dominare
        # i moltissimi negative pair.
        loss = (
            0.5
            *
            positive_loss
            +
            0.5
            *
            negative_loss
        )


        stats = {

            "positive_mean":
                positive_distances
                .mean()
                .detach(),

            "negative_mean":
                negative_distances
                .mean()
                .detach(),

            "positive_loss":
                positive_loss.detach(),

            "negative_loss":
                negative_loss.detach()
        }


        return (
            loss,
            stats
        )


contrastive_criterion = (
    BalancedInBatchContrastiveLoss(
        margin=1.25
    )
)

In [20]:
# ============================================================
# CELL 21 — HYBRID CONTRASTIVE + BATCH-HARD TRIPLET
# ============================================================

TRIPLET_WEIGHT = 0.25


class HybridMetricLoss(nn.Module):

    def __init__(
        self,
        contrastive_criterion,
        triplet_criterion,
        triplet_weight=0.25
    ):

        super().__init__()

        self.contrastive = (
            contrastive_criterion
        )

        self.triplet = (
            triplet_criterion
        )

        self.triplet_weight = float(
            triplet_weight
        )


    def forward(
        self,
        embeddings,
        labels
    ):

        (
            contrastive_loss,
            contrastive_stats
        ) = self.contrastive(
            embeddings,
            labels
        )


        (
            triplet_loss,
            triplet_stats
        ) = self.triplet(
            embeddings,
            labels
        )


        total_loss = (
            contrastive_loss
            +
            self.triplet_weight
            *
            triplet_loss
        )


        stats = {

            "contrastive_loss":
                contrastive_loss.detach(),

            "triplet_loss":
                triplet_loss.detach(),

            "hard_positive":
                triplet_stats[
                    "hard_positive"
                ],

            "hard_negative":
                triplet_stats[
                    "hard_negative"
                ],

            "separation":
                triplet_stats[
                    "separation"
                ],

            "active_fraction":
                triplet_stats[
                    "active_fraction"
                ],

            "positive_mean":
                contrastive_stats[
                    "positive_mean"
                ],

            "negative_mean":
                contrastive_stats[
                    "negative_mean"
                ]
        }


        return (
            total_loss,
            stats
        )


hybrid_criterion = HybridMetricLoss(

    contrastive_criterion=
        contrastive_criterion,

    triplet_criterion=
        triplet_criterion,

    triplet_weight=
        TRIPLET_WEIGHT
)


print(
    "Triplet weight:",
    TRIPLET_WEIGHT
)

Triplet weight: 0.25


In [21]:
# ============================================================
# CELL 22 — FRESH OPTIMIZER FOR HYBRID FINE-TUNING
# ============================================================

HYBRID_LEARNING_RATE = 5e-5


optimizer = torch.optim.AdamW(

    triplet_model.parameters(),

    lr=
        HYBRID_LEARNING_RATE,

    weight_decay=
        WEIGHT_DECAY,

    fused=
        DEVICE.type == "cuda"
)


scaler = torch.amp.GradScaler(

    "cuda",

    enabled=
        AMP_ENABLED
)


print(
    "Hybrid LR:",
    HYBRID_LEARNING_RATE
)

Hybrid LR: 5e-05


In [22]:
# ============================================================
# CELL 23 — TRAIN ONE HYBRID EPOCH
# ============================================================

def train_hybrid_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler
):

    model.train()

    total_loss = 0.0

    total_contrastive_loss = 0.0
    total_triplet_loss = 0.0

    total_positive_mean = 0.0
    total_negative_mean = 0.0

    total_hard_positive = 0.0
    total_hard_negative = 0.0

    total_separation = 0.0
    total_active_fraction = 0.0

    total_batches = 0

    start_time = time.perf_counter()


    for batch in loader:

        x = (
            batch["x"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        labels = (
            batch["class_id"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        # ====================================================
        # FORWARD
        # ====================================================

        with torch.autocast(

            device_type=
                DEVICE.type,

            dtype=
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16,

            enabled=
                AMP_ENABLED

        ):

            embeddings = model(
                x
            )


        # Loss in FP32 internamente
        (
            loss,
            stats
        ) = criterion(
            embeddings,
            labels
        )


        # ====================================================
        # BACKWARD
        # ====================================================

        if AMP_ENABLED:

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            loss.backward()

            optimizer.step()


        # ====================================================
        # METRICHE
        # ====================================================

        total_loss += (
            loss.detach().item()
        )

        total_contrastive_loss += (
            stats[
                "contrastive_loss"
            ].item()
        )

        total_triplet_loss += (
            stats[
                "triplet_loss"
            ].item()
        )

        total_positive_mean += (
            stats[
                "positive_mean"
            ].item()
        )

        total_negative_mean += (
            stats[
                "negative_mean"
            ].item()
        )

        total_hard_positive += (
            stats[
                "hard_positive"
            ].item()
        )

        total_hard_negative += (
            stats[
                "hard_negative"
            ].item()
        )

        total_separation += (
            stats[
                "separation"
            ].item()
        )

        total_active_fraction += (
            stats[
                "active_fraction"
            ].item()
        )

        total_batches += 1


    elapsed = (
        time.perf_counter()
        -
        start_time
    )


    return {

        "loss":
            total_loss
            /
            total_batches,

        "contrastive_loss":
            total_contrastive_loss
            /
            total_batches,

        "triplet_loss":
            total_triplet_loss
            /
            total_batches,

        "positive_mean":
            total_positive_mean
            /
            total_batches,

        "negative_mean":
            total_negative_mean
            /
            total_batches,

        "hard_positive":
            total_hard_positive
            /
            total_batches,

        "hard_negative":
            total_hard_negative
            /
            total_batches,

        "separation":
            total_separation
            /
            total_batches,

        "active_fraction":
            total_active_fraction
            /
            total_batches,

        "seconds":
            elapsed
    }

In [23]:
# ============================================================
# CELL 24 — HYBRID SMOKE TEST
# ============================================================

HYBRID_SMOKE_EPOCHS = 3


# ============================================================
# BACKUP DEL MODELLO EUCLIDEAN PULITO
# ============================================================

hybrid_initial_model_state = copy.deepcopy(
    triplet_model.state_dict()
)


hybrid_initial_optimizer_state = copy.deepcopy(
    optimizer.state_dict()
)


print("=" * 82)
print("HYBRID CONTRASTIVE + BATCH-HARD TRIPLET — SMOKE TEST")
print("=" * 82)

print(
    "Contrastive margin:",
    contrastive_criterion.margin
)

print(
    "Triplet margin:",
    triplet_criterion.margin
)

print(
    "Triplet weight:",
    TRIPLET_WEIGHT
)

print(
    "Learning rate:",
    HYBRID_LEARNING_RATE
)

print()


for epoch in range(
    1,
    HYBRID_SMOKE_EPOCHS + 1
):

    # ========================================================
    # TRAIN
    # ========================================================

    train_metrics = (
        train_hybrid_epoch(

            model=
                triplet_model,

            loader=
                train_loader,

            criterion=
                hybrid_criterion,

            optimizer=
                optimizer,

            scaler=
                scaler
        )
    )


    # ========================================================
    # MULTICLASS VALIDATION
    # ========================================================

    (
        val_metrics,
        _,
        _
    ) = evaluate_multiclass_metric_space(

        model=
            triplet_model,

        reference_loader=
            reference_loader,

        val_loader=
            val_loader
    )


    print(

        f"Epoch {epoch:02d}/{HYBRID_SMOKE_EPOCHS}"

        f" | total "
        f"{train_metrics['loss']:.4f}"

        f" | ctr "
        f"{train_metrics['contrastive_loss']:.4f}"

        f" | tri "
        f"{train_metrics['triplet_loss']:.4f}"

        f" | pos "
        f"{train_metrics['positive_mean']:.4f}"

        f" | neg "
        f"{train_metrics['negative_mean']:.4f}"

        f" | hard+ "
        f"{train_metrics['hard_positive']:.4f}"

        f" | hard- "
        f"{train_metrics['hard_negative']:.4f}"

        f" | sep "
        f"{train_metrics['separation']:.4f}"

        f" | active "
        f"{train_metrics['active_fraction'] * 100:.1f}%"

        f" | val Acc "
        f"{val_metrics['accuracy']:.4f}"

        f" | val F1 "
        f"{val_metrics['macro_f1']:.4f}"

        f" | bal Acc "
        f"{val_metrics['balanced_accuracy']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

HYBRID CONTRASTIVE + BATCH-HARD TRIPLET — SMOKE TEST
Contrastive margin: 1.25
Triplet margin: 0.2
Triplet weight: 0.25
Learning rate: 5e-05

Epoch 01/3 | total 0.5117 | ctr 0.3561 | tri 0.6225 | pos 0.5010 | neg 0.6076 | hard+ 0.6587 | hard- 0.2362 | sep -0.4225 | active 100.0% | val Acc 0.2385 | val F1 0.1963 | bal Acc 0.2472 | 7.6s
Epoch 02/3 | total 0.5037 | ctr 0.3603 | tri 0.5733 | pos 0.5089 | neg 0.5956 | hard+ 0.6410 | hard- 0.2677 | sep -0.3733 | active 100.0% | val Acc 0.2367 | val F1 0.1961 | bal Acc 0.2460 | 7.9s
Epoch 03/3 | total 0.4996 | ctr 0.3605 | tri 0.5563 | pos 0.5192 | neg 0.6010 | hard+ 0.6437 | hard- 0.2875 | sep -0.3563 | active 100.0% | val Acc 0.2380 | val F1 0.1994 | bal Acc 0.2461 | 6.9s
